In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [72]:
# data1 = pd.read_csv('D:\Desktop\安卓38关之后闯关.csv',index_col =False)
# data2 = pd.read_csv('D:\Desktop\安卓38关之后付费.csv',index_col =False)
data = pd.read_csv('D:\Desktop/1207-0224用户关卡付费.csv',index_col=False)
data.sort_values(by=['$ssid','$sts'],ascending=[True,True],inplace=True)
# 填充nan
data['level_id'] = data['level_id'].fillna(method='ffill')
data.fillna(value=0,inplace=True)
data = data.query('level_id>=38')

In [73]:
data.head()

,$ssid,$sts,level_id,#em_revenue_ex
356783,10O2qWBNd2cib37mrA3mTB,1733594041467,151.0,0.0
356802,10O2qWBNd2cib37mrA3mTB,1733594210484,150.0,0.0
333850,10O2qWBNd2cib37mrA3mTB,1733722477493,148.0,0.0
398724,10O2qWBNd2cib37mrA3mTB,1733802814722,150.0,0.0
398751,10O2qWBNd2cib37mrA3mTB,1733802937112,151.0,0.0


In [74]:
# 筛选首次进入关卡的时间
data = data.groupby(by=['$ssid','level_id','#em_revenue_ex'],as_index=False)['$sts'].min()

data.head()

,$ssid,level_id,#em_revenue_ex,$sts
0,10O2qWBNd2cib37mrA3mTB,148.0,0.0,1733722477493
1,10O2qWBNd2cib37mrA3mTB,149.0,0.0,1733803178212
2,10O2qWBNd2cib37mrA3mTB,150.0,0.0,1733594210484
3,10O2qWBNd2cib37mrA3mTB,151.0,0.0,1733594041467
4,10R8X699GXMV3rvfkXcVMC,40.0,0.0,1734014060513


In [75]:
def func1(data):
    # 初始指针 和用户id
    index_lev  =  38
    ssid = data.iloc[0,:]['$ssid']
    for  i in range(data.shape[0]):
        # 更新用户id user_ssid
        df=data.iloc[i,:]
        user_ssid = df['$ssid']
        if ssid ==user_ssid :
            if index_lev == df['level_id']:
                # 如果关卡id均指向初始关卡则更新付费关卡为初始关卡
                data.loc[i,'付费关卡'] = index_lev
                # print(f'----初始关卡付费，更新第{i}行',data.iloc[i,:]['付费关卡'] )
                
            else:
                # 如果关卡id不指向初始关卡则更新关卡id,后更新付费关卡
                index_lev = df['level_id']
                data.loc[i,'付费关卡'] = index_lev
                # print(f'----非初始关卡付费，更新关卡和第{i}行',data.iloc[i,:]['付费关卡'] )
        else:
            # 如果用户id发生改变则重新初始化关卡id和用户id
            ssid = data.iloc[i,:]['$ssid']
            index_lev = 38
            data.loc[i,'付费关卡'] = index_lev
            # print(f'----非初始用户、初始关卡付费，更新用户第{i}行',data.iloc[i,:]['付费关卡'] )
    return data

In [76]:
data=func1(data)

In [77]:
data.head()

,$ssid,level_id,#em_revenue_ex,$sts,付费关卡
0,10O2qWBNd2cib37mrA3mTB,148.0,0.0,1733722477493,148.0
1,10O2qWBNd2cib37mrA3mTB,149.0,0.0,1733803178212,149.0
2,10O2qWBNd2cib37mrA3mTB,150.0,0.0,1733594210484,150.0
3,10O2qWBNd2cib37mrA3mTB,151.0,0.0,1733594041467,151.0
4,10R8X699GXMV3rvfkXcVMC,40.0,0.0,1734014060513,38.0


In [78]:
data['付费id'] = data.apply(lambda x:x['$ssid'] if x['#em_revenue_ex']>0 else np.nan,axis=1)

In [79]:
res1 = data.groupby(by=['付费关卡'],as_index = False)[['$ssid','付费id','#em_revenue_ex']].agg({'$ssid':'nunique','付费id':'nunique','#em_revenue_ex':'sum'})

In [80]:
res1['付费率'] = res1['付费id']/res1['$ssid']
res1['arppu'] = res1['#em_revenue_ex']/res1['付费id']

In [81]:
res1.head()

,付费关卡,$ssid,付费id,#em_revenue_ex,付费率,arppu
0,38.0,2166,137,1722.441740,0.063250,12.572567
1,39.0,1697,73,1461.000250,0.043017,20.013702
2,40.0,1760,260,4448.462895,0.147727,17.109473
3,41.0,1684,60,1335.798400,0.035629,22.263307
4,42.0,1665,176,3603.063845,0.105706,20.471954


In [82]:
res1.to_csv('../临时文件/更新后38后关付费率.csv')